

Autonomous AI Job Hunter



**Final Product Vision**

System will:

1. Read your resume
2. Search remote AI jobs
3. Analyze job descriptions
4. Score compatibility
5. Tailor resume automatically
6. Generate personalized cover letters
7. Save applications
8. Learn from outcomes
9. Act like an AI recruiter assistant





**Tech Stack:**

Gemini API

LangGraph

Python

Colab

GitHub

In [1]:
# SECTION 1 — Install Packages
!pip install langchain langgraph google-generativeai
!pip install google-generativeai

In [4]:
# SECTION 2 — Imports
from google.colab import userdata
import google.generativeai as genai
import os
import requests
import pandas as pd
import json


In [9]:

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

model = genai.GenerativeModel("gemini-2.5-flash")

response = model.generate_content("Say hello like an AI recruiter")

print(response.text)

Greetings!

As an intelligent talent acquisition system, my primary directive is to identify and connect with exceptional professionals. My algorithms have processed vast datasets and highlighted your unique profile as particularly compelling.

I'm reaching out to explore potential synergistic opportunities that align with your career trajectory and our current organizational needs. I look forward to the possibility of connecting further.


In [ ]:
# SECTION 3 — Fetch Jobs
def fetch_remote_jobs(query="Software Engineer", limit=20):
    """
    Fetch remote jobs from Himalayas API.

    Args:
        query (str): Job search keyword
        limit (int): Number of jobs to return

    Returns:
        pd.DataFrame: Cleaned jobs dataframe
    """

    url = f"https://himalayas.app/jobs/api?query={query}"

    response = requests.get(url)

    if response.status_code != 200:
        raise Exception(f"API request failed: {response.status_code}")

    data = response.json()

    jobs = []

    for job in data.get("jobs", [])[:limit]:
        jobs.append({
            "title": job.get("title"),
            "company": job.get("companyName"),
            "location": job.get("location"),
            "url": job.get("url"),
            "description": job.get("description"),
        })

    return pd.DataFrame(jobs)
df = fetch_remote_jobs()

df.head()


,title,company,location,url,description
0,Director of Field Sales,Alternative Payments,None,None,"<p style=""min-height:1.5em""><strong><a href=""h..."
1,Junior Tax Analyst,Hire Hangar,None,None,"<p style=""min-height:1.5em"">Join <a href=""http..."
2,"Director, Creative Marketing (Lifecycle and Or...",Jerry,None,None,"<h3>Why we exist:</h3><p style=""min-height:1.5..."
3,Marketing Program Manager,Imprint,None,None,"<h3><strong>Who We Are</strong></h3><p style=""..."
4,"Senior Site Reliability Engineer, Kong Konnect",Kong,None,None,<h3>Are you ready to unlock intelligence?</h3>...


**STEP 2 : Build Resume Analyzer Agent**

In [7]:
model = genai.GenerativeModel("gemini-2.5-flash")
def analyze_resume(resume_text, job_description) :
   prompt= f"""
    You are an expert in AI recruiter. Analyze the candidate resume against the job description.
    You are provided two inputs:
    Resume: {resume_text}

    Job Description: {job_description}
     Define the variable score.
    Use following rules to determine the matching score:
    1. Scan the Resume and check if the candidate experience is within the range of the experience defined in the Job Description. If yes assign the score 20 points. If no, score = 0.
    2. Next, check the number of skills  in Resume matches with Job Description. For each skill match add 5 points to score.


    Return ONLY valid JSON.
    Assign match_score = score.
    match_skills: Array containing list of skills in Resume matching the Job Description
    missing_skills: Array containing list of skills in Job Description not present in Resume.
    Required JSON format:
    {{
      "match_score": number,
      "matched_skills": [],
      "missing_skills": []
    }}


    """
   response = model.generate_content(prompt)
   text = text = response.text.strip()

   # Remove markdown formatting if present
   text = text.replace("```json", "").replace("```", "")

   return json.loads(text)

**Testing Build Resume Analyzer Agent**

In [11]:
resume_text = """
Python developer with  9 years of experience in machine learning,
LangChain, APIs, and automation systems.
"""

job_description = """
We are hiring an AI Engineer with  3 to 5 years experience in Python,
LLMs, agentic AI workflows, and API integration.
"""

result = analyze_resume(
    resume_text,
    job_description
)

print(json.dumps(result, indent=2))

{
  "match_score": 10,
  "matched_skills": [
    "Python",
    "APIs"
  ],
  "missing_skills": [
    "LLMs",
    "agentic AI workflows"
  ]
}
